In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

explanation_text = """
<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #dee2e6; font-size: 13px;">
<b>Complex Pole-Zero & Temporal Evolution Explorer</b><br>
* <b>Generalization:</b> Explores complex conjugate pole pairs $p = x \\pm j y$.<br>
* <b>Model:</b> $h[n] = 2 r^n \\cos(\\omega n) u[n]$ where $r = \\sqrt{x^2+y^2}$ (radius) and $\\omega = \\text{atan2}(y,x)$ (frequency).<br>
* <b>Stability:</b> $r < 1$ (Stable inside unit circle) | $r > 1$ (Unstable outside). Use the sliders below to interact.
</div>
"""

display(widgets.HTML(explanation_text))

out = widgets.Output()

n_samples = 40
n_vec = np.arange(n_samples)

def plot_complex_evolution(real_val, imag_val):
    with out:
        clear_output(wait=True)
        
        # Double the width of the right figure (e.g., left width = 5.5, right width = 11 -> total figsize width = 16.5)
        fig, (ax_pz, ax_time) = plt.subplots(1, 2, figsize=(16.5, 5), gridspec_kw={'width_ratios': [1, 2]})
        plt.subplots_adjust(wspace=0.25)

        # --- 1. Pole-Zero Map ---
        ax_pz.set_aspect('equal')
        ax_pz.set_xlim(-1.8, 1.8)
        ax_pz.set_ylim(-1.8, 1.8)
        ax_pz.axhline(0, color='black', linewidth=1)
        ax_pz.axvline(0, color='black', linewidth=1)
        ax_pz.grid(True, linestyle=':', alpha=0.7)

        theta = np.linspace(0, 2*np.pi, 100)
        ax_pz.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.5, label='Unit Circle')

        ax_pz.scatter([real_val], [imag_val], s=120, color='r', marker='x', linewidths=3, label='Pole p')
        if imag_val != 0:
            # Conjugate pole plotted with a green 'x' as requested
            ax_pz.scatter([real_val], [-imag_val], s=120, color='g', marker='x', linewidths=3, label='Conjugate p*')

        ax_pz.scatter([0], [0], s=100, facecolors='none', edgecolors='b', linewidths=2, marker='o', label='Zero at origin')

        r = np.sqrt(real_val**2 + imag_val**2)
        stability = "Stable" if r < 1 else ("Unstable" if r > 1 else "Critically Stable")
        
        ax_pz.set_title(f'Pole-Zero Map ({stability}, r={r:.2f})', fontsize=10, fontweight='bold')
        ax_pz.set_xlabel('Real Part', fontsize=9)
        ax_pz.set_ylabel('Imaginary Part', fontsize=9)
        
        # Move legend below the left plot to avoid overlapping with poles
        ax_pz.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=8)

        # --- 2. Time Domain Plot ---
        omega = np.arctan2(imag_val, real_val)
        h_n = 2 * (r**n_vec) * np.cos(omega * n_vec)

        ax_time.stem(n_vec, h_n, linefmt='r-', markerfmt='ro', basefmt='k-')
        ax_time.set_title(f'Temporal Evolution: $h[n] = 2 r^n \\cos(\\omega n) u[n]$', fontsize=10, fontweight='bold')
        ax_time.set_xlabel('Time index n', fontsize=9)
        ax_time.set_ylabel('h[n]', fontsize=9)
        ax_time.set_xlim(-1, n_samples)
        
        # Adjusted y-limits dynamically to prevent clipping, capped cleanly
        max_abs_val = np.max(np.abs(h_n))
        y_limit = max(1.5, min(max_abs_val * 1.25, 50.0))
        ax_time.set_ylim(-y_limit, y_limit)
        ax_time.grid(True, linestyle=':', alpha=0.7)

        plt.show()

real_slider = widgets.FloatSlider(value=0.5, min=-1.5, max=1.5, step=0.01, description='Real:', style={'description_width': 'initial'})
imag_slider = widgets.FloatSlider(value=0.5, min=-1.5, max=1.5, step=0.01, description='Imag:', style={'description_width': 'initial'})

plot_complex_evolution(real_slider.value, imag_slider.value)

interactive_plot = widgets.interactive(plot_complex_evolution, real_val=real_slider, imag_val=imag_slider)
display(widgets.VBox([interactive_plot, out]))